# Testing for differences in power scaling according to different contexts

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import f

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import f_oneway as ftest

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'allan_var.tsv'

PARAMETERS_PATH = '../_ignore/4-complexity_tests/allan-variance/av-params.csv'
# PARAMETERS_PATH = './ignore/av-params-20250825.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'context_type'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    result = str(x)
    
    if '-DISPEL-' in result:
        result = 'DISPEL'
    
    if result.startswith('GCSAusE'):
        result = 'GCSAusE'
    
    if result.startswith('se'):
        result = 'CORAAL'
        
    if result.startswith('call'):
        result = x.split('-')[0]
    
    if result.startswith('MICASE'):
        result = 'MICASE'
    
    if result.startswith('instruction'):
        result = x.split('-')[1]
    
    if result.startswith('CABNC'):
        result = 'CABNC'
        
    if result.startswith('SBCSAE'):
        result = 'SBCSAE'
        
    if result.startswith('SCoSE'):
        result = 'SCoSE'
        
    if result.startswith('candor'):
        result = 'CANDOR'
    
    if result.startswith('grace'):
        result = 'Miao-fNIRS'
    
    return result

In [5]:
def context(x):
    context = None
    if 'callfriend' in x:
        context = 'friend'

    if 'callhome' in x:
        context = 'family'

    if ('frederiksen' in x.lower()) or ('graesser' in x.lower()):
        context = 'tutoring'

    if 'disepel' in x.lower():
        context = 'gaming tutorial'

    if 'med_school' in x.lower():
        context = 'group learning'

    if 'MICASE' in x:
        context = x.split('-')[1]

    if ('candor' in x.lower()) or ('grace' in x.lower()):
        context = 'first meeting'

    return context

#### Loading Results

In [6]:
results = pd.read_csv(PARAMETERS_PATH)

In [7]:
results.isna().sum()

x_turn_id            0
null                 0
self                 0
conversation_id      0
speaker              0
slope              843
b0                 751
dtype: int64

In [8]:
results.isna().mean()

x_turn_id          0.000000
null               0.000000
self               0.000000
conversation_id    0.000000
speaker            0.000000
slope              0.000741
b0                 0.000660
dtype: float64

In [9]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

(np.int64(7424), np.float64(0.00652730495456208))

In [10]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0
0,CABNC-0missing-CABNC-0missing-KB0RE004-10,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.183603,-11.122748
1,CABNC-0missing-CABNC-0missing-KB0RE004-11,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.979970,-6.513821
2,CABNC-0missing-CABNC-0missing-KB0RE004-12,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.234872,-5.934323
3,CABNC-0missing-CABNC-0missing-KB0RE004-13,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,0.019517,-10.000541
4,CABNC-0missing-CABNC-0missing-KB0RE004-15,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-0.995607,-9.275234


In [11]:
sel = results['slope'].isna() | (results['slope'].abs() == np.inf) 
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(8267), np.float64(0.007268484652392876), np.int64(1129109))

In [12]:
results['self'].value_counts()

self
True     572655
False    564721
Name: count, dtype: int64

In [13]:
results['x_turn_id'].loc[sel].nunique()

8153

In [14]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
is_null = results['null']
is_null.sum()

np.int64(99856)

Adding Corpus names back in . . .

In [15]:
# results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]
results[subcorpora_col] = [context(x) for x in tqdm(results['conversation_id'])]

  0%|          | 0/1137376 [00:00<?, ?it/s]

In [16]:
results[subcorpora_col].value_counts()

context_type
family                   79159
first meeting            51846
les                      40465
friend                   36373
office_hours             30178
study_groups             29279
lel                      24490
seminars                 20137
labs                     17377
student_presentations    17018
colloquia                16132
discussion               10903
meetings                 10754
service_encounters        7070
advising                  6534
defense                   5917
tours                     3280
interviews                2201
tutoring                   995
group learning             361
Name: count, dtype: int64

In [17]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0,context_type
0,CABNC-0missing-CABNC-0missing-KB0RE004-10,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.183603,-11.122748,None
1,CABNC-0missing-CABNC-0missing-KB0RE004-11,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.979970,-6.513821,None
2,CABNC-0missing-CABNC-0missing-KB0RE004-12,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.234872,-5.934323,None
3,CABNC-0missing-CABNC-0missing-KB0RE004-13,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,0.019517,-10.000541,None
4,CABNC-0missing-CABNC-0missing-KB0RE004-15,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-0.995607,-9.275234,None


### Removing null values and unknown context types

In [18]:
results = results.loc[
    (~is_null)
    & (~results[subcorpora_col].isna())
    & (~sel)
]
results.shape

(372813, 8)

## Differences between context types

In [19]:
subcorpora_col = 'speaker'

### Self

In [20]:
sel_ = results['self']
sel_.sum()

np.int64(190583)

In [21]:
# means
df = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('mean').reset_index()

# var
df['s2'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('var').reset_index()['slope']

# wj for welch
df['wj'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: len(x)/x.var()).reset_index()['slope']

# n
df['n'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('count').reset_index()['slope']

# SSW
df['ssw'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: ((x-x.mean())**2).sum()).reset_index()['slope']

# SSB
df['ssb'] = (df['slope'] - df['slope'].mean())**2

In [22]:
df.head(n=100)

,speaker,slope,s2,wj,n,ssw,ssb
0,558a327cfdf99b2d75651681,0.102515,1.293841,26.278335,34,42.696768,0.098367
1,56df9a836b4093000bd896c7,-0.145403,1.443087,69.988851,101,144.308699,0.004319
2,57d6dd8d6598aa000199188b,-0.102316,1.083707,137.491060,149,160.388610,0.011838
3,5aa61dd36475f90001a05dcf,-0.134759,1.240459,141.076761,175,215.839942,0.005831
4,5aeb922a75deca0001624d23,-0.330970,1.310968,63.311991,83,107.499384,0.014364
...,...,...,...,...,...,...,...
95,MICASE-colloquia-MICASE-colloquia-col285mx038-S9,4.670575,35.644166,0.056110,2,35.644166,23.830951
96,MICASE-colloquia-MICASE-colloquia-col385mu054-S2,-0.386836,1.924390,12.471483,24,44.260975,0.030876
97,MICASE-colloquia-MICASE-colloquia-col385mu054-S3,-0.164819,0.463949,605.669705,281,129.905788,0.002144
98,MICASE-colloquia-MICASE-colloquia-col425mx075-S1,-0.950316,3.045696,2.298325,7,18.274175,0.546409


And the actual F-test calculations

In [23]:
df = df.loc[df['n'] > 2]

In [24]:
k = len(df)
w = df['wj'].sum()
mu_ = ((df['wj'] * df['slope'])/w).sum()

In [25]:
df['inv_nj'] = 1/(df['n']-1)
df['p_wj'] = (1 - (df['wj']/w))**2

In [26]:
num = (1/(k-1)) * df['wj'] * ((df['slope']-mu_)**2)
num = num.sum()

In [27]:
denom = 1 + ((2*(k-2))/((k**2)-1)) * (df['inv_nj'] * df['p_wj']).sum()

In [28]:
dof = ((k**2)-1) / (3 * (df['inv_nj'] * df['p_wj']).sum())
dof

np.float64(8594.854549228558)

In [29]:
F = num/denom
F

np.float64(2.1366156505497287)

In [30]:
F, k-1, dof, 1-f.cdf(F,k-1,dof)

(np.float64(2.1366156505497287),
 1514,
 np.float64(8594.854549228558),
 np.float64(1.1102230246251565e-16))

### Other

Are there individual differences in the degree to which individual speakers influence their conversation partners? We replicate the same procedure in the "self" approach above on data for exponential decay rate ($\alpha$) for the influence of an utterance on the "other".

In [31]:
sel_ = ~results['self']
sel_.sum()

np.int64(182230)

In [32]:
# means
df = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('mean').reset_index()

# var
df['s2'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('var').reset_index()['slope']

# wj for welch
# df['wj'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: len(x)/x.var()).reset_index()['slope']

# n
df['n'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg('count').reset_index()['slope']

# wj for welch
df['wj'] = df['n'] / df['s2']

# SSW
df['ssw'] = results[[subcorpora_col, 'slope']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: ((x-x.mean())**2).sum()).reset_index()['slope']

# SSB
df['ssb'] = (df['slope'] - (df['slope'].mean()))**2

In [33]:
df.head(n=100)

,speaker,slope,s2,n,wj,ssw,ssb
0,558a327cfdf99b2d75651681,-0.272879,1.190264,33,27.724946,38.088442,0.005260
1,56df9a836b4093000bd896c7,-0.075521,1.729263,103,59.562962,176.384781,0.015583
2,57d6dd8d6598aa000199188b,-0.087462,0.780051,148,189.731178,114.667501,0.012745
3,5aa61dd36475f90001a05dcf,-0.151019,1.348342,175,129.789039,234.611491,0.002434
4,5aeb922a75deca0001624d23,-0.031849,1.216870,84,69.029587,101.000169,0.028394
...,...,...,...,...,...,...,...
95,MICASE-colloquia-MICASE-colloquia-col200mx133-S1,0.223018,0.591918,32,54.061531,18.349462,0.179244
96,MICASE-colloquia-MICASE-colloquia-col200mx133-S10,-0.359287,1.253002,2,1.596167,1.253002,0.025260
97,MICASE-colloquia-MICASE-colloquia-col200mx133-S2,-0.631095,0.320015,13,40.623141,3.840176,0.185538
98,MICASE-colloquia-MICASE-colloquia-col200mx133-S3,-0.333141,0.420069,337,802.249853,141.143061,0.017632


In [34]:
df.isna().sum()

speaker      0
slope        0
s2         244
n            0
wj         244
ssw          0
ssb          0
dtype: int64

And the actual F-test calculations

In [35]:
print(df.shape)
df = df.loc[df['n'] > 2]
df.shape

(2295, 7)


(1885, 7)

In [36]:
k, w, mu_ = len(df), df['wj'].sum(), ((df['wj'] * df['slope'])/w).sum()
k, w, mu_

(1885, np.float64(245407.50148932548), np.float64(-0.11432672826330145))

In [37]:
df['inv_nj'] = 1/(df['n']-1)
df['p_wj'] = (1 - (df['wj']/w))**2

In [38]:
num = (1/(k-1)) * df['wj'] * ((df['slope']-mu_)**2)
num = num.sum()
num

np.float64(7.62681067417846)

In [39]:
denom = 1 + ((2*(k-2))/((k**2)-1)) * (df['inv_nj'] * df['p_wj']).sum()
denom

np.float64(1.1894978897699264)

In [40]:
dof = ((k**2)-1) / (3 * (df['inv_nj'] * df['p_wj']).sum())

In [41]:
F = num/denom
F

np.float64(6.411789999605332)

In [42]:
F, k-1, dof, 1-f.cdf(F,k-1,dof)

(np.float64(6.411789999605332),
 1884,
 np.float64(6624.524077062081),
 np.float64(1.1102230246251565e-16))